# 07 - Authentication and JWT

## 1. Authentication vs Authorization

| | Authentication (authn) | Authorization (authz) |
|---|---|---|
| Question | **Who are you?** | **What are you allowed to do?** |
| Checked by | Login, token | Role, permissions |
| Failure code | **401** Unauthorized | **403** Forbidden |
| Example | No token sent | Customer calls an admin endpoint |

**Real-world example:** At an airport, showing your ID is authentication. Your boarding pass only letting you onto your own flight is authorization.

## 2. What is a JWT?

**Definition:** A JWT (JSON Web Token) is a signed token the server gives after login. The client sends it on every request in the header:
Authorization: Bearer <token>

**Three parts, separated by dots:** header.payload.signature
- **Header:** algorithm used (e.g. HS256)
- **Payload:** claims like user id, role, expiry (exp)
- **Signature:** made with a secret key; proves the token wasn't changed

**Important:** The payload is **encoded, not encrypted**. Anyone can read it, but nobody can change it without breaking the signature. **Never put passwords or secrets in the payload.**

**Interview line:** "Authentication checks who you are and fails with 401; authorization checks what you can do and fails with 403. A JWT has a header, payload and signature, and the payload is readable but tamper-proof."

In [ ]:
# ==================================================
# 1. JWT FROM SCRATCH - readable but tamper-proof
# ==================================================
import base64, json, hmac, hashlib, time

SECRET = "my-server-secret"

def b64(data):
    return base64.urlsafe_b64encode(json.dumps(data).encode()).rstrip(b"=").decode()

def sign(msg):
    sig = hmac.new(SECRET.encode(), msg.encode(), hashlib.sha256).digest()
    return base64.urlsafe_b64encode(sig).rstrip(b"=").decode()

def create_token(payload):
    head = b64({"alg": "HS256", "typ": "JWT"})
    body = b64(payload)
    return f"{head}.{body}.{sign(head + '.' + body)}"

def verify_token(token):
    head, body, sig = token.split(".")
    if not hmac.compare_digest(sig, sign(head + "." + body)):
        return 401, "Invalid signature (token was changed)"
    payload = json.loads(base64.urlsafe_b64decode(body + "=" * (-len(body) % 4)))
    if payload["exp"] < time.time():
        return 401, "Token expired"
    return 200, payload

token = create_token({"user_id": 5, "role": "customer", "exp": time.time() + 900})
print("=" * 55)
print("[TOKEN]   ", token[:60], "...")

# Anyone can READ the payload (encoded, not encrypted)
body = token.split(".")[1]
print("[DECODED] ", json.loads(base64.urlsafe_b64decode(body + "=" * (-len(body) % 4))))

# Valid token
print("[VERIFY]  ", verify_token(token))

# Attacker changes role to admin
head, _, sig = token.split(".")
fake_body = b64({"user_id": 5, "role": "admin", "exp": time.time() + 900})
print("[TAMPERED]", verify_token(f"{head}.{fake_body}.{sig}"))
print("=" * 55)

## 3. Access Token vs Refresh Token

| | Access token | Refresh token |
|---|---|---|
| Purpose | Call protected APIs | Get a new access token |
| Lifetime | Short (e.g. 15 minutes) | Long (e.g. 7 days) |
| Sent with | Every API request | Only to /auth/refresh |
| If stolen | Damage limited by short expiry | Can be revoked on the server |

**Flow:**
1. User logs in → server returns access token + refresh token
2. Client calls APIs with the access token
3. Access token expires → API returns 401
4. Client sends refresh token to /auth/refresh → gets a new access token
5. On logout, the refresh token is revoked

**Real-world example (my project):** Azure Entra ID issues the JWT when the user signs in. Our backend never creates tokens; it only **verifies** the signature, expiry and roles on each request.

**Interview line:** "Access tokens are short-lived and sent on every request; refresh tokens are long-lived and only used to get new access tokens. In my project, Entra ID issues the JWT and our backend only verifies it."

In [ ]:
# ==================================================
# 2. ACCESS + REFRESH TOKEN FLOW
# ==================================================
import time, secrets

ACCESS_TTL = 2          # seconds (15 min in real life)
access_tokens = {}      # token -> (user_id, expiry)
refresh_tokens = {}     # token -> user_id

def login(user_id):
    access = secrets.token_hex(8)
    refresh = secrets.token_hex(8)
    access_tokens[access] = (user_id, time.time() + ACCESS_TTL)
    refresh_tokens[refresh] = user_id
    return access, refresh

def call_api(access):
    if access not in access_tokens:
        return 401, "Invalid token"
    user_id, expiry = access_tokens[access]
    if time.time() > expiry:
        return 401, "Access token expired"
    return 200, f"Orders for user {user_id}"

def refresh(refresh_token):
    if refresh_token not in refresh_tokens:
        return 401, "Refresh token revoked"
    user_id = refresh_tokens[refresh_token]
    new_access = secrets.token_hex(8)
    access_tokens[new_access] = (user_id, time.time() + ACCESS_TTL)
    return 200, new_access

def logout(refresh_token):
    refresh_tokens.pop(refresh_token, None)

print("=" * 55)
access, ref = login(5)
print("[1. LOGIN]    got access + refresh token")
print("[2. CALL]    ", call_api(access))
time.sleep(ACCESS_TTL + 1)
print("[3. EXPIRED] ", call_api(access))
code, access = refresh(ref)
print("[4. REFRESH]  new access token issued")
print("[5. CALL]    ", call_api(access))
logout(ref)
print("[6. LOGOUT]  ", refresh(ref))
print("=" * 55)

## Key Takeaways

- Authentication = who you are (401); authorization = what you can do (403)
- JWT = header.payload.signature, sent as Authorization: Bearer <token>
- Payload is **encoded, not encrypted**: readable but tamper-proof; no secrets inside
- Access token: short-lived, every request. Refresh token: long-lived, only to renew
- In my project, Entra ID issues the JWT; the backend only verifies it